# 04 - Gold Load Staging

**Layer:** Gold (SQL)

## Goal
Load the cleaned Silver Excel file into the `stg_airbnb` staging table in SQL Server.
Normalization into the final tables is done in SQL scripts afterwards.

## Step 1: Connection Settings
Ask for the server, database, and ODBC driver. Windows authentication is used.

In [8]:
import os
import pandas as pd
from urllib.parse import quote_plus
from sqlalchemy import create_engine
from sqlalchemy import text


In [6]:
server = input("SQL Server name (example: localhost\\SQLEXPRESS): ").strip()
database = input("Database name [AirbnbEurope]: ").strip()
if database == "":
    database = "AirbnbEurope"

default_driver = "ODBC Driver 17 for SQL Server"
driver = input(f"ODBC driver [{default_driver}]: ").strip()
if driver == "":
    driver = default_driver

conn_str = (
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
    "Trusted_Connection=yes;"
)
engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(conn_str),
    fast_executemany=True,
)

# Test the connection
with engine.connect() as connection:
    print("Connected successfully")

Connected successfully


## Step 2: Read the Silver File

In [7]:
default_silver = "../data/silver"
silver_path = input(f"Enter silver folder path [{default_silver}]: ").strip()
if silver_path == "":
    silver_path = default_silver

df = pd.read_excel(os.path.join(silver_path, "airbnb_clean.xlsx"))
print("Loaded shape:", df.shape)

Loaded shape: (51707, 26)


## Step 3: Load into the Staging Table
Insert the data in chunks. The table is emptied first so the notebook can be re-run safely.

In [9]:
# Empty the staging table before loading
with engine.begin() as connection:
    connection.execute(text("TRUNCATE TABLE stg_airbnb"))

chunk_size = 5000
start = 0
while start < len(df):
    chunk = df.iloc[start:start + chunk_size]
    chunk.to_sql("stg_airbnb", engine, if_exists="append", index=False)
    start += chunk_size
    print(f"Loaded {min(start, len(df))} / {len(df)}")

Loaded 5000 / 51707
Loaded 10000 / 51707
Loaded 15000 / 51707
Loaded 20000 / 51707
Loaded 25000 / 51707
Loaded 30000 / 51707
Loaded 35000 / 51707
Loaded 40000 / 51707
Loaded 45000 / 51707
Loaded 50000 / 51707
Loaded 51707 / 51707


## Step 4: Verify the Load

In [10]:
with engine.connect() as connection:
    sql_count = connection.execute(text("SELECT COUNT(*) FROM stg_airbnb")).scalar()

print("Rows in SQL:", sql_count)
print("Rows in file:", len(df))
print("Match:", sql_count == len(df))

Rows in SQL: 51707
Rows in file: 51707
Match: True
